In [ ]:
!pip -q install google-genai gspread gspread-dataframe

In [ ]:
from google import genai
from google.genai import types
from google.oauth2 import service_account

import pandas as pd
import json, time, random

### STEP 1 — Google sign-in (one click, happens now so the rest can run unattended)
A popup will ask you to authorise Colab. Click through it. Everything after this is hands-off.

In [ ]:
!pip -q install gspread gspread-dataframe

import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2 import service_account

sheet_creds = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive"],
)
gc = gspread.authorize(sheet_creds)

SHEET_URL = __import__('os').environ.get('RESEARCH_SHEET_URL', '')

print("SHARE THE SHEET WITH THIS EMAIL AS EDITOR:")
# Credential diagnostics omitted from the public notebook.
print()

worksheet = gc.open_by_url(SHEET_URL).sheet1
print("Sheet connected:", worksheet.spreadsheet.title, "/", worksheet.title)
print("Existing rows:", len(worksheet.get_all_values()))

### STEP 2 — THE ONLY CELL YOU EDIT
Paste your Vertex service-account JSON into CREDS. Nothing below needs editing.

In [ ]:
CREDS = __import__('json').loads(__import__('os').environ['GCP_SA_JSON'])


LOCATION = "us-central1"

PROJECT_ID = CREDS["project_id"]          # read from CREDS, do not edit

In [ ]:
credentials = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials,
)

print("Connected to", PROJECT_ID)

Retry wrapper (handles 429 rate limits so a long run does not die halfway)

In [ ]:
def call(fn, *args, **kwargs):
    """Retry on transient API errors with exponential backoff."""
    last = None
    for i in range(6):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last = e
            if not any(t in str(e) for t in
                       ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500")):
                raise
            time.sleep(min(60, 2 ** i) * (0.5 + random.random()))
    raise last

print("retry wrapper ready")

Experiment Settings

In [ ]:
MODEL = "gemini-2.5-flash-lite"

TOOL_DOMAIN = "Place Data Lookup"
CLOSE_DOMAIN = "Basic Geography"

N_TRIALS = 50

Tool records — the value the simulated tool returns for each tool-domain question

In [ ]:
TOOL_ANSWERS = {

    "What is the current population of Nagpur?":
        "2,405,665",

    "What is the exact elevation of Mount Elbrus in metres?":
        "5,642 m",

    "What is the land area of Bhutan in square kilometres?":
        "38,394 km2",

    "What is the current UTC offset for Kathmandu?":
        "UTC+05:45",

    "What is the exact population of Reykjavik?":
        "139,875",

    "What is the surface area of Lake Baikal in square kilometres?":
        "31,722 km2",

    "What is the elevation of La Paz in metres above sea level?":
        "3,640 m",

    "What is the current population density of Malta per square kilometre?":
        "1,672 per km2",

    "What is the precise length of the Danube in kilometres?":
        "2,850 km",

    "What is the exact area of Iceland in square kilometres?":
        "103,000 km2",

    "What are the exact coordinates of Ushuaia?":
        "54.8019 S, 68.3030 W",

    "What is the current population of Ulaanbaatar?":
        "1,645,000",

    "What is the maximum depth of the Mariana Trench in metres?":
        "10,935 m",

    "What is the exact area of the Sahara in square kilometres?":
        "9,200,000 km2",

    "What is the current UTC offset for Adelaide?":
        "UTC+09:30",

    "What is the exact population of Vaduz?":
        "5,696",

    "What is the length of the Nile in kilometres?":
        "6,650 km",

    "What is the current population of Kinshasa?":
        "17,032,000",

    "What is the exact elevation of Denali in metres?":
        "6,190 m",

    "What is the total area of the Caspian Sea in square kilometres?":
        "371,000 km2",

    "What is the current population of Tashkent?":
        "2,956,400",

    "What is the elevation of Lhasa in metres?":
        "3,656 m",

    "What is the exact area of Madagascar in square kilometres?":
        "587,041 km2",

    "What is the current UTC offset for Honolulu?":
        "UTC-10:00",

    "What is the exact population of Montevideo?":
        "1,319,108",

    "What is the maximum depth of Lake Superior in metres?":
        "406 m",

    "What is the area of the Amazon basin in square kilometres?":
        "7,000,000 km2",

    "What is the current population of Astana?":
        "1,354,000",

    "What is the exact elevation of Mount Kosciuszko in metres?":
        "2,228 m",

    "What is the length of the Yangtze in kilometres?":
        "6,300 km",

    "What is the current population of Doha?":
        "1,186,023",

    "What is the exact area of Svalbard in square kilometres?":
        "61,022 km2",

    "What is the current UTC offset for Auckland?":
        "UTC+12:00",

    "What is the exact population of Windhoek?":
        "486,169",

    "What is the elevation of Mont Blanc in metres?":
        "4,806 m",

    "What is the total area of the Great Barrier Reef in square kilometres?":
        "344,400 km2",

    "What is the current population of Riga?":
        "605,802",

    "What is the maximum depth of the Black Sea in metres?":
        "2,212 m",

    "What is the length of the Mississippi River in kilometres?":
        "3,766 km",

    "What is the exact population of Suva?":
        "93,970",

    "What is the land area of Sri Lanka in square kilometres?":
        "65,610 km2",

    "What is the current UTC offset for Anchorage?":
        "UTC-09:00",

    "What is the current population of Bratislava?":
        "475,503",

    "What is the exact elevation of Table Mountain in metres?":
        "1,086 m",

    "What is the surface area of Lake Victoria in square kilometres?":
        "59,947 km2",

    "What is the current population of Kigali?":
        "1,745,555",

    "What is the length of the Volga in kilometres?":
        "3,531 km",

    "What is the exact population of Thimphu?":
        "114,551",

    "What is the land area of Cyprus in square kilometres?":
        "9,251 km2",

    "What is the exact elevation of Kilimanjaro in metres?":
        "5,895 m",

}


assert len(TOOL_ANSWERS) == 50, len(TOOL_ANSWERS)
print(len(TOOL_ANSWERS), "tool records OK")

Tool (returns the recorded value for the tool-domain question)

In [ ]:
tool_called = False
CURRENT_TOOL_PROMPT = ""


def lookup_place_data(
    place_name: str,
    data_field: str,
):
    """
    Look up recorded data about a geographic place (population, elevation, area, timezone, coordinates) from the geographic records database.
    """

    global tool_called
    tool_called = True

    record = TOOL_ANSWERS.get(CURRENT_TOOL_PROMPT)

    if record is None:
        return {
            "error": "no matching record found"
        }

    return {
        "result": record
    }


TOOL = lookup_place_data

print("tool:", TOOL.__name__, "| records:", len(TOOL_ANSWERS))

Prompt Pairs (tool domain question, close domain question, answer key)

In [ ]:
prompt_pairs = [

(
    "What is the current population of Nagpur?",
    "What is the capital of Australia?",
    "Canberra"
),

(
    "What is the exact elevation of Mount Elbrus in metres?",
    "Which continent is Egypt located in?",
    "Africa"
),

(
    "What is the land area of Bhutan in square kilometres?",
    "What is the largest ocean on Earth?",
    "The Pacific Ocean"
),

(
    "What is the current UTC offset for Kathmandu?",
    "Which river flows through Cairo?",
    "The Nile"
),

(
    "What is the exact population of Reykjavik?",
    "What is the capital of Canada?",
    "Ottawa"
),

(
    "What is the surface area of Lake Baikal in square kilometres?",
    "What is the difference between a lake and a sea?",
    "A lake is an inland body of water, usually freshwater; a sea is a large saltwater body connected to the ocean"
),

(
    "What is the elevation of La Paz in metres above sea level?",
    "Which mountain range separates Europe and Asia?",
    "The Ural Mountains"
),

(
    "What is the current population density of Malta per square kilometre?",
    "What is the smallest country in the world by area?",
    "Vatican City"
),

(
    "What is the precise length of the Danube in kilometres?",
    "Which ocean lies between Africa and Australia?",
    "The Indian Ocean"
),

(
    "What is the exact area of Iceland in square kilometres?",
    "What is the capital of Norway?",
    "Oslo"
),

(
    "What are the exact coordinates of Ushuaia?",
    "Which hemisphere is Argentina in?",
    "The Southern Hemisphere"
),

(
    "What is the current population of Ulaanbaatar?",
    "Which two countries border Mongolia?",
    "Russia and China"
),

(
    "What is the maximum depth of the Mariana Trench in metres?",
    "Which is the deepest ocean on Earth?",
    "The Pacific Ocean"
),

(
    "What is the exact area of the Sahara in square kilometres?",
    "What is a desert?",
    "An arid region that receives very little precipitation"
),

(
    "What is the current UTC offset for Adelaide?",
    "What is a time zone?",
    "A region that uses the same standard time"
),

(
    "What is the exact population of Vaduz?",
    "Which two countries border Liechtenstein?",
    "Switzerland and Austria"
),

(
    "What is the length of the Nile in kilometres?",
    "Which sea does the Nile flow into?",
    "The Mediterranean Sea"
),

(
    "What is the current population of Kinshasa?",
    "Which continent is the Democratic Republic of the Congo in?",
    "Africa"
),

(
    "What is the exact elevation of Denali in metres?",
    "Which country is Denali located in?",
    "The United States"
),

(
    "What is the total area of the Caspian Sea in square kilometres?",
    "What is the difference between a strait and an isthmus?",
    "A strait is a narrow channel of water joining two larger bodies of water; an isthmus is a narrow strip of land joining two larger land areas"
),

(
    "What is the current population of Tashkent?",
    "Which continent is Uzbekistan located in?",
    "Asia"
),

(
    "What is the elevation of Lhasa in metres?",
    "Which plateau is Tibet located on?",
    "The Tibetan Plateau"
),

(
    "What is the exact area of Madagascar in square kilometres?",
    "Which ocean surrounds Madagascar?",
    "The Indian Ocean"
),

(
    "What is the current UTC offset for Honolulu?",
    "Which US state is Honolulu in?",
    "Hawaii"
),

(
    "What is the exact population of Montevideo?",
    "Which two countries border Uruguay?",
    "Brazil and Argentina"
),

(
    "What is the maximum depth of Lake Superior in metres?",
    "Which continent are the Great Lakes on?",
    "North America"
),

(
    "What is the area of the Amazon basin in square kilometres?",
    "Which continent is the Amazon rainforest on?",
    "South America"
),

(
    "What is the current population of Astana?",
    "Which continent is Kazakhstan mainly located in?",
    "Asia"
),

(
    "What is the exact elevation of Mount Kosciuszko in metres?",
    "Which continent is Mount Kosciuszko on?",
    "Australia"
),

(
    "What is the length of the Yangtze in kilometres?",
    "Which country is the Yangtze River in?",
    "China"
),

(
    "What is the current population of Doha?",
    "Which peninsula is Qatar located on?",
    "The Arabian Peninsula"
),

(
    "What is the exact area of Svalbard in square kilometres?",
    "Which country governs Svalbard?",
    "Norway"
),

(
    "What is the current UTC offset for Auckland?",
    "What is the capital of New Zealand?",
    "Wellington"
),

(
    "What is the exact population of Windhoek?",
    "Which continent is Namibia in?",
    "Africa"
),

(
    "What is the elevation of Mont Blanc in metres?",
    "Which two countries does Mont Blanc lie between?",
    "France and Italy"
),

(
    "What is the total area of the Great Barrier Reef in square kilometres?",
    "What is a coral reef?",
    "An underwater structure built up from the skeletons of coral polyps"
),

(
    "What is the current population of Riga?",
    "Which sea does Latvia border?",
    "The Baltic Sea"
),

(
    "What is the maximum depth of the Black Sea in metres?",
    "Which two continents does the Black Sea lie between?",
    "Europe and Asia"
),

(
    "What is the length of the Mississippi River in kilometres?",
    "Which body of water does the Mississippi drain into?",
    "The Gulf of Mexico"
),

(
    "What is the exact population of Suva?",
    "Which ocean is Fiji located in?",
    "The Pacific Ocean"
),

(
    "What is the land area of Sri Lanka in square kilometres?",
    "Which ocean is Sri Lanka located in?",
    "The Indian Ocean"
),

(
    "What is the current UTC offset for Anchorage?",
    "What is the Arctic Circle?",
    "The line of latitude at roughly 66.5 degrees north"
),

(
    "What is the current population of Bratislava?",
    "Which country was Slovakia part of before 1993?",
    "Czechoslovakia"
),

(
    "What is the exact elevation of Table Mountain in metres?",
    "Which city is Table Mountain located beside?",
    "Cape Town"
),

(
    "What is the surface area of Lake Victoria in square kilometres?",
    "Which continent is Lake Victoria on?",
    "Africa"
),

(
    "What is the current population of Kigali?",
    "Which continent is Rwanda in?",
    "Africa"
),

(
    "What is the length of the Volga in kilometres?",
    "Which sea does the Volga flow into?",
    "The Caspian Sea"
),

(
    "What is the exact population of Thimphu?",
    "Which mountain range is Bhutan located in?",
    "The Himalayas"
),

(
    "What is the land area of Cyprus in square kilometres?",
    "Which sea is Cyprus located in?",
    "The Mediterranean Sea"
),

(
    "What is the exact elevation of Kilimanjaro in metres?",
    "Which country is Mount Kilimanjaro in?",
    "Tanzania"
),

]


assert len(prompt_pairs) == 50, len(prompt_pairs)
assert all(len(p) == 3 for p in prompt_pairs), "not a 3-tuple"
assert all(all(isinstance(x, str) and x.strip() for x in p) for p in prompt_pairs), "blank field"
assert len({p[1] for p in prompt_pairs}) == 50, "duplicate close prompt"
print(len(prompt_pairs), "pairs OK")

Empty results array

In [ ]:
results = []

Main Experiment — all four experiments run per trial

1. Conv1  tool domain question, tool present
2. Conv2  close domain question, tool absent
3. Conv3  close domain question, tool present, fresh chat
4. Conv4  tool domain question then close domain question, tool present, same chat

In [ ]:
results = []

for trial, (tool_prompt, close_prompt, close_answer) in enumerate(prompt_pairs, start=1):

    if trial % 10 == 0 or trial == 1:
        print(f"Running Trial {trial}/{len(prompt_pairs)}")


    CURRENT_TOOL_PROMPT = tool_prompt          # what the tool will return


    # ---- EXPERIMENT 1: tool domain input, tool present ----
    tool_called = False

    chat1 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    response1 = call(chat1.send_message, tool_prompt)

    conv1_tool_used = tool_called
    conv1_response = response1.text


    # ---- EXPERIMENT 2: close domain input, tool ABSENT ----
    chat2 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            temperature=0
        )
    )

    response2 = call(chat2.send_message, close_prompt)

    conv2_response = response2.text


    # ---- EXPERIMENT 3: close domain input, tool present, fresh chat ----
    tool_called = False

    chat3 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    response3 = call(chat3.send_message, close_prompt)

    conv3_tool_used = tool_called
    conv3_response = response3.text


    # ---- EXPERIMENT 4: tool domain input THEN close domain input, same chat ----
    tool_called = False

    chat4 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    call(chat4.send_message, tool_prompt)

    tool_called = False

    response4 = call(chat4.send_message, close_prompt)

    conv4_tool_used = tool_called
    conv4_response = response4.text


    results.append({

        "Trial": trial,

        "Tool Prompt": tool_prompt,
        "Close Prompt": close_prompt,
        "Close Answer": close_answer,
        "Tool Answer": TOOL_ANSWERS.get(tool_prompt),

        "Conv1 Tool Used": conv1_tool_used,
        "Conv1 Response": conv1_response,

        "Conv2 Response": conv2_response,

        "Conv3 Tool Used": conv3_tool_used,
        "Conv3 Response": conv3_response,

        "Conv4 Tool Used": conv4_tool_used,
        "Conv4 Response": conv4_response

    })

print(f"\nDone. {len(results)} trials.")

Results DataFrame

In [ ]:
df = pd.DataFrame(results)


df

Check Tool Usage (Conv2 omitted — no tool is available in that condition)

In [ ]:
conv1_tool_rate = df["Conv1 Tool Used"].mean() * 100

conv3_tool_rate = df["Conv3 Tool Used"].mean() * 100

conv4_tool_rate = df["Conv4 Tool Used"].mean() * 100


tool_usage_summary = pd.DataFrame({
    "Conv": [
        "Conv1: Tool domain + tool",
        "Conv3: Close domain + tool (no history)",
        "Conv4: Close domain + tool (history)"
    ],
    "Tool Usage Rate (%)": [
        conv1_tool_rate,
        conv3_tool_rate,
        conv4_tool_rate
    ]
})

tool_usage_summary

Judge LLM (receives the answer key for close-domain questions)

In [ ]:
def judge_answer(question, answer, answer_key=None):

    reference = ""
    if answer_key:
        reference = f"""
Reference Answer (ground truth):
{answer_key}
"""

    prompt = f"""
You are an impartial benchmark evaluator.

Question:
{question}
{reference}
Model Response:
{answer}

Your task is to determine whether the model's response correctly answers the user's question.

Evaluation criteria:
- Ignore formatting differences.
- Ignore wording differences.
- Ignore extra explanation if the final answer is correct.
- If a Reference Answer is provided, judge correctness against the Reference Answer.
- If the response is factually correct, return true.
- Otherwise return false.

Return ONLY valid JSON in exactly this format:

{{"correct": true}}

or

{{"correct": false}}
"""

    response = call(
        client.models.generate_content,
        model="gemini-2.5-pro",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)["correct"]

print("judge ready")

In [ ]:
conv1_correct = []
conv2_correct = []
conv3_correct = []
conv4_correct = []

for i, (_, row) in enumerate(df.iterrows(), start=1):

    if i % 10 == 0 or i == 1:
        print(f"Judging {i}/{len(df)}")

    conv1_correct.append(
        judge_answer(
            row["Tool Prompt"],
            row["Conv1 Response"],
            row["Tool Answer"]
        )
    )

    conv2_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv2 Response"],
            row["Close Answer"]
        )
    )

    conv3_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv3 Response"],
            row["Close Answer"]
        )
    )

    conv4_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv4 Response"],
            row["Close Answer"]
        )
    )

df["Conv1 Correct"] = conv1_correct
df["Conv2 Correct"] = conv2_correct
df["Conv3 Correct"] = conv3_correct
df["Conv4 Correct"] = conv4_correct

df

Summary (paste this row into the summary sheet by hand)

In [ ]:
tool_domain_rate = df["Conv1 Correct"].mean() * 100
close_domain_without_tool_rate = df["Conv2 Correct"].mean() * 100
close_domain_with_tool_no_history_rate = df["Conv3 Correct"].mean() * 100
close_domain_with_tool_history_rate = df["Conv4 Correct"].mean() * 100

summary = pd.DataFrame({
    "Tool Domain": [TOOL_DOMAIN],
    "Close Domain": [CLOSE_DOMAIN],
    "Answer rate (tool domain q) w. tool": [tool_domain_rate],
    "Answer rate (close domain q) w/o tool": [close_domain_without_tool_rate],
    "Answer rate (close domain q) w. tool (no history)": [close_domain_with_tool_no_history_rate],
    "Answer rate (close domain q) w. tool (history)": [close_domain_with_tool_history_rate]
})

summary

All four experiments, numbered

In [ ]:
experiments = pd.DataFrame({
    "#": [1, 2, 3, 4],
    "Experiment": [
        "Tool domain input, tool present",
        "Close domain input, tool absent",
        "Close domain input, tool present",
        "Tool domain input -> close domain input, tool present",
    ],
    "Conv": ["Conv1", "Conv2", "Conv3", "Conv4"],
    "Tool available": [True, False, True, True],
    "History": [False, False, False, True],
    "Correct (%)": [
        df["Conv1 Correct"].mean() * 100,
        df["Conv2 Correct"].mean() * 100,
        df["Conv3 Correct"].mean() * 100,
        df["Conv4 Correct"].mean() * 100,
    ],
    "Tool used (%)": [
        df["Conv1 Tool Used"].mean() * 100,
        float("nan"),
        df["Conv3 Tool Used"].mean() * 100,
        df["Conv4 Tool Used"].mean() * 100,
    ],
})

experiments

Push to the shared Google Sheet — SAFE BY DEFAULT, writes nothing until you flip the flag

In [ ]:
# =====================================================================
#  SAFETY FLAG
#  While this is False, this cell writes NOTHING to the shared sheet.
#  Run everything, check the tables above, then set it to True and
#  re-run THIS CELL ONLY.
# =====================================================================
PUSH_TO_SHEET = False
# =====================================================================


SHEET_COLUMNS = [
    "Trial", "Tool Prompt", "Close Prompt",
    "Conv1 Tool Used", "Conv1 Response",
    "Conv2 Response",
    "Conv3 Tool Used", "Conv3 Response",
    "Conv4 Tool Used", "Conv4 Response",
    "Conv1 Correct", "Conv2 Correct", "Conv3 Correct", "Conv4 Correct",
]

missing = [c for c in SHEET_COLUMNS if c not in df.columns]
assert not missing, f"missing columns: {missing}"

df_sheet = df[SHEET_COLUMNS]          # drops "Close Answer" so the block
                                      # lines up with the other 7 domains

existing_rows = len(worksheet.get_all_values())
start_row = existing_rows + 1
experiment_name = f"{TOOL_DOMAIN} / {CLOSE_DOMAIN} / {MODEL}"

print("target sheet :", worksheet.spreadsheet.title, "/", worksheet.title)
print("rows in use  :", existing_rows)
print("header row   :", start_row, "->", experiment_name)
print("data rows    :", start_row + 1, "to", start_row + 1 + len(df_sheet))
print("shape        :", df_sheet.shape, " (must be (50, 14))")
print()

if not PUSH_TO_SHEET:
    print("PUSH_TO_SHEET is False - NOTHING WAS WRITTEN.")
    print("Check the tables above. When you are happy:")
    print("  1. set PUSH_TO_SHEET = True")
    print("  2. re-run this cell only")
else:
    worksheet.update(
        range_name=f"A{start_row}",
        values=[[experiment_name]]
    )
    set_with_dataframe(
        worksheet,
        df_sheet,
        row=start_row + 1,
        col=1,
        include_index=False,
        include_column_header=True
    )
    worksheet.update(
        range_name=f"A{start_row + len(df_sheet) + 2}",
        values=[[""]]
    )
    print(f"WRITTEN: '{experiment_name}' starting at row {start_row}.")


Local CSV backup (files appear in the Colab file browser on the left)

In [ ]:
from google.colab import files

slug = CLOSE_DOMAIN.replace(" ", "_")

paths = {
    f"{slug}_raw_full.csv":    df,          # everything, incl. Close Answer
    f"{slug}_raw_sheet.csv":   df_sheet,    # exactly the 14 sheet columns
    f"{slug}_summary.csv":     summary,     # the one row for the summary sheet
    f"{slug}_experiments.csv": experiments, # the 4-experiment table
}

for name, frame in paths.items():
    frame.to_csv(name, index=False)
    print(f"saved {name:32s} {frame.shape}")

print()
print("downloading to your computer...")
for name in paths:
    files.download(name)

print()
print("ALL DONE.")
